# AI-Based Golf Coach - End-to-End Colab Prototype

**Author/Student:** Isaiah Goh  
**Mentor:** Dr. Qingyang Xiao  
**Prototype version:** 1.0  
**Notebook date:** July 25, 2026

This single notebook implements a runnable prototype that:

1. Collects a golfer profile and calculates BMI.
2. Accepts uploaded golf videos such as MP4, WMV, HEVC, AVI, MOV, and MKV.
3. Converts the video to a Colab-friendly MP4 when needed.
4. Detects and annotates 33 body landmarks with MediaPipe Pose Landmarker.
5. Calculates joint angles, posture, balance, head stability, rotation, tempo, and arm-extension indicators.
6. Segments the swing into address, backswing, top, downswing, impact, and follow-through.
7. Demonstrates supervised machine learning with a Random Forest.
8. Demonstrates deep learning with a GRU sequence model.
9. Demonstrates reinforcement learning with a feedback-updated contextual bandit.
10. Produces coaching suggestions, a four-week practice plan, figures, CSV/JSON/HTML reports, model files, an annotated video, and one ZIP package.

> **Important limitation:** This is an educational MVP, not a clinically validated health system or a professionally validated golf-swing grading product. The included ML and DL training data are synthetic unless a real labeled dataset is supplied. Scores must not be presented as certified coaching judgments.

## How to run

1. Open this notebook in Google Colab.
2. Select **Runtime > Run all**.
3. The default `VIDEO_MODE = "demo"` completes the entire pipeline without requiring an upload.
4. For a real swing, change `VIDEO_MODE` to `"upload"`, rerun from the configuration cell, and upload one video.
5. Keep the full golfer visible, use stable lighting, and place the camera on a tripod when possible.
6. Download the final ZIP package at the end.

For optional GPU acceleration of the GRU model, select **Runtime > Change runtime type > GPU**. The notebook also works on CPU.

In [ ]:
# Install only the packages that may not already be available in Colab.
# Colab commonly includes NumPy, pandas, scikit-learn, PyTorch, Matplotlib, and OpenCV.
%pip -q install mediapipe joblib

In [ ]:
from __future__ import annotations

import base64
import copy
import html
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import time
import urllib.request
import warnings
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
from IPython.display import HTML, Markdown, Video, display
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

try:
    import mediapipe as mp
except Exception as exc:
    mp = None
    print("MediaPipe import failed. Rerun the installation cell, then restart the runtime if needed.")
    print("Import error:", repr(exc))

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("MediaPipe:", getattr(mp, "__version__", "not available"))
print("Colab detected:", IN_COLAB)
print("CUDA available:", torch.cuda.is_available())

## 1. Configuration

Use `demo` for a guaranteed end-to-end test. Change to `upload` for a real video or `path` for an existing file already mounted in Colab.

In [ ]:
# ---------------------------- USER CONFIGURATION ----------------------------
VIDEO_MODE = "demo"          # "demo", "upload", or "path"
LOCAL_VIDEO_PATH = ""        # Used only when VIDEO_MODE == "path"

MAX_SECONDS = 30              # Long uploads are truncated for this MVP
FRAME_STRIDE = 2              # Analyze every Nth frame; skipped frames reuse the last pose for annotation
MAX_OUTPUT_WIDTH = 960        # Resize very large videos to reduce runtime and memory

AUTO_DOWNLOAD_FINAL_ZIP = False   # Set True to trigger a Colab browser download at the end
INCLUDE_HEALTH_NOTES_IN_REPORT = False
RUN_DEEP_LEARNING = True
DL_EPOCHS = 18

BASE_DIR = Path("/content" if IN_COLAB else Path.cwd())
PROJECT_DIR = BASE_DIR / "isaiah_goh_ai_golf_coach"
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = PROJECT_DIR / "models"
for directory in (PROJECT_DIR, OUTPUT_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

POSE_MODEL_PATH = MODEL_DIR / "pose_landmarker_lite.task"
POSE_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
    "pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
)

print("Project directory:", PROJECT_DIR)

## 2. Golfer profile and privacy choices

The profile supports personalization. Health information is optional and should be minimized. This notebook processes files inside the active runtime and does not call a third-party generative AI API.

In [ ]:
# Replace the example values before processing a real user.
USER_PROFILE: Dict[str, Any] = {
    "name": "Demo Golfer",
    "gender": "Prefer not to say",
    "age": 18,
    "height_cm": 175.0,
    "weight_kg": 70.0,
    "golf_handedness": "Right-handed",
    "experience_level": "Beginner",  # Beginner, Intermediate, Advanced
    "gym_activity_days_per_week": 3,
    "primary_goal": "Improve swing consistency and balance",
    "health_notes": "None reported",
    "consent_to_process_video": True,
}


def calculate_bmi(weight_kg: float, height_cm: float) -> float:
    height_m = max(float(height_cm) / 100.0, 0.01)
    return float(weight_kg) / (height_m ** 2)

USER_PROFILE["bmi"] = round(
    calculate_bmi(USER_PROFILE["weight_kg"], USER_PROFILE["height_cm"]), 1
)

if not USER_PROFILE.get("consent_to_process_video", False):
    raise PermissionError("Video-processing consent is required before running this prototype.")

profile_for_display = {
    key: value for key, value in USER_PROFILE.items()
    if key not in {"health_notes", "consent_to_process_video"}
}
display(pd.DataFrame([profile_for_display]).T.rename(columns={0: "Value"}))
print("Health notes are used only for cautious wording and are excluded from the report by default.")

## 3. Pose, geometry, video, and demo utilities

The demo path generates a synthetic stick-figure swing with known landmarks. The upload path uses MediaPipe Pose Landmarker in video mode. MediaPipe provides 33 body landmarks; this prototype derives golf-related indicators from them.

In [ ]:
# MediaPipe/BlazePose landmark indices.
NOSE = 0
LEFT_SHOULDER, RIGHT_SHOULDER = 11, 12
LEFT_ELBOW, RIGHT_ELBOW = 13, 14
LEFT_WRIST, RIGHT_WRIST = 15, 16
LEFT_HIP, RIGHT_HIP = 23, 24
LEFT_KNEE, RIGHT_KNEE = 25, 26
LEFT_ANKLE, RIGHT_ANKLE = 27, 28
LEFT_HEEL, RIGHT_HEEL = 29, 30
LEFT_FOOT, RIGHT_FOOT = 31, 32

POSE_CONNECTIONS: List[Tuple[int, int]] = [
    (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10), (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21),
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22),
    (11, 23), (12, 24), (23, 24), (23, 25), (25, 27), (27, 29), (29, 31),
    (24, 26), (26, 28), (28, 30), (30, 32), (27, 31), (28, 32)
]


def clamp(value: float, low: float, high: float) -> float:
    return float(max(low, min(high, value)))


def smoothstep(x: float) -> float:
    x = clamp(x, 0.0, 1.0)
    return x * x * (3.0 - 2.0 * x)


def lerp(a: np.ndarray, b: np.ndarray, u: float) -> np.ndarray:
    return a + (b - a) * smoothstep(u)


def point(landmarks: np.ndarray, idx: int) -> np.ndarray:
    return np.asarray(landmarks[idx, :2], dtype=float)


def midpoint(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return (np.asarray(a, dtype=float) + np.asarray(b, dtype=float)) / 2.0


def euclidean(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(np.asarray(a, dtype=float) - np.asarray(b, dtype=float)))


def angle_three_points(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """Returns angle ABC in degrees from 0 to 180."""
    ba = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    bc = np.asarray(c, dtype=float) - np.asarray(b, dtype=float)
    denom = np.linalg.norm(ba) * np.linalg.norm(bc)
    if denom < 1e-9:
        return float("nan")
    cosine = float(np.dot(ba, bc) / denom)
    return float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))


def line_angle_degrees(a: np.ndarray, b: np.ndarray) -> float:
    delta = np.asarray(b, dtype=float) - np.asarray(a, dtype=float)
    return float(np.degrees(np.arctan2(delta[1], delta[0])))


def wrapped_angle_difference(a: float, b: float) -> float:
    difference = (a - b + 180.0) % 360.0 - 180.0
    return abs(float(difference))


def draw_pose(frame: np.ndarray, landmarks: Optional[np.ndarray],
              phase: str = "", overall_score: Optional[float] = None,
              extra_text: Optional[Sequence[str]] = None) -> np.ndarray:
    canvas = frame.copy()
    height, width = canvas.shape[:2]
    if landmarks is not None:
        for start, end in POSE_CONNECTIONS:
            if landmarks[start, 3] < 0.25 or landmarks[end, 3] < 0.25:
                continue
            p1 = (int(landmarks[start, 0] * width), int(landmarks[start, 1] * height))
            p2 = (int(landmarks[end, 0] * width), int(landmarks[end, 1] * height))
            cv2.line(canvas, p1, p2, (80, 220, 80), 2, cv2.LINE_AA)
        for idx, lm in enumerate(landmarks):
            if lm[3] < 0.25:
                continue
            center = (int(lm[0] * width), int(lm[1] * height))
            radius = 5 if idx in {LEFT_WRIST, RIGHT_WRIST, LEFT_HIP, RIGHT_HIP} else 3
            cv2.circle(canvas, center, radius, (40, 80, 240), -1, cv2.LINE_AA)

    overlay_lines = []
    if phase:
        overlay_lines.append(f"Phase: {phase}")
    if overall_score is not None:
        overlay_lines.append(f"Prototype score: {overall_score:.1f}/100")
    if extra_text:
        overlay_lines.extend([str(x) for x in extra_text])

    if overlay_lines:
        box_height = 34 + 25 * len(overlay_lines)
        cv2.rectangle(canvas, (8, 8), (min(width - 8, 480), box_height), (0, 0, 0), -1)
        for i, text in enumerate(overlay_lines):
            cv2.putText(
                canvas, text, (18, 38 + 25 * i), cv2.FONT_HERSHEY_SIMPLEX,
                0.62, (255, 255, 255), 2, cv2.LINE_AA
            )
    return canvas


def run_ffmpeg(command: Sequence[str]) -> bool:
    result = subprocess.run(
        list(command), stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    if result.returncode != 0:
        print("ffmpeg warning:", result.stderr[-800:])
        return False
    return True


def standardize_video(input_path: Path, output_path: Path, max_width: int = 960) -> Path:
    """Convert common input formats to H.264 MP4, preserving audio when possible."""
    command = [
        "ffmpeg", "-y", "-i", str(input_path),
        "-vf", f"scale='min({max_width},iw)':-2",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "23",
        "-pix_fmt", "yuv420p", "-c:a", "aac", "-movflags", "+faststart",
        str(output_path)
    ]
    if run_ffmpeg(command) and output_path.exists():
        return output_path
    print("Video conversion failed; attempting to use the original file directly.")
    return input_path


def download_pose_model(model_path: Path = POSE_MODEL_PATH) -> Path:
    if model_path.exists() and model_path.stat().st_size > 100_000:
        return model_path
    print("Downloading MediaPipe pose model...")
    urllib.request.urlretrieve(POSE_MODEL_URL, model_path)
    return model_path

In [ ]:
def demo_landmarks(frame_index: int, total_frames: int) -> np.ndarray:
    """Generate a plausible front-view swing path for an executable demo."""
    t = frame_index / max(total_frames - 1, 1)
    lm = np.zeros((33, 5), dtype=np.float32)
    lm[:, 3] = 1.0  # visibility
    lm[:, 4] = 1.0  # presence

    # Piecewise hand path: address -> top -> impact -> finish.
    address = np.array([0.56, 0.57])
    top = np.array([0.32, 0.25])
    impact = np.array([0.58, 0.58])
    finish = np.array([0.78, 0.28])
    if t < 0.14:
        hand = address + np.array([0.005 * math.sin(t * 40), 0.0])
        phase_strength = 0.0
    elif t < 0.45:
        u = (t - 0.14) / 0.31
        hand = lerp(address, top, u)
        phase_strength = smoothstep(u)
    elif t < 0.62:
        u = (t - 0.45) / 0.17
        hand = lerp(top, impact, u)
        phase_strength = 1.0 - smoothstep(u)
    else:
        u = (t - 0.62) / 0.38
        hand = lerp(impact, finish, u)
        phase_strength = -smoothstep(u)

    sway = 0.075 * math.sin(math.pi * t)
    head_sway = 0.050 * math.sin(2 * math.pi * t)
    shoulder_tilt = 0.035 * phase_strength
    hip_tilt = 0.014 * phase_strength

    # Face landmarks.
    lm[NOSE, :2] = [0.50 + sway + head_sway, 0.20]
    lm[1, :2] = [0.485 + sway, 0.19]
    lm[2, :2] = [0.475 + sway, 0.19]
    lm[3, :2] = [0.465 + sway, 0.195]
    lm[4, :2] = [0.515 + sway, 0.19]
    lm[5, :2] = [0.525 + sway, 0.19]
    lm[6, :2] = [0.535 + sway, 0.195]
    lm[7, :2] = [0.45 + sway, 0.205]
    lm[8, :2] = [0.55 + sway, 0.205]
    lm[9, :2] = [0.485 + sway, 0.225]
    lm[10, :2] = [0.515 + sway, 0.225]

    left_shoulder = np.array([0.42 + sway, 0.36 - shoulder_tilt])
    right_shoulder = np.array([0.58 + sway, 0.36 + shoulder_tilt])
    left_hip = np.array([0.45 + sway, 0.61 - hip_tilt])
    right_hip = np.array([0.55 + sway, 0.61 + hip_tilt])
    lm[LEFT_SHOULDER, :2] = left_shoulder
    lm[RIGHT_SHOULDER, :2] = right_shoulder
    lm[LEFT_HIP, :2] = left_hip
    lm[RIGHT_HIP, :2] = right_hip

    # Both hands stay close on the club grip.
    left_wrist = hand + np.array([-0.012, 0.003])
    right_wrist = hand + np.array([0.012, -0.003])
    lm[LEFT_WRIST, :2] = left_wrist
    lm[RIGHT_WRIST, :2] = right_wrist
    lm[17, :2] = left_wrist + [-0.010, 0.0]
    lm[19, :2] = left_wrist + [-0.005, -0.006]
    lm[21, :2] = left_wrist + [0.004, 0.004]
    lm[18, :2] = right_wrist + [0.010, 0.0]
    lm[20, :2] = right_wrist + [0.005, -0.006]
    lm[22, :2] = right_wrist + [-0.004, 0.004]

    # Elbows follow the grip with controlled bend.
    left_elbow = 0.56 * left_shoulder + 0.44 * left_wrist + np.array([-0.015, 0.02])
    right_elbow = 0.56 * right_shoulder + 0.44 * right_wrist + np.array([0.015, 0.02])
    lm[LEFT_ELBOW, :2] = left_elbow
    lm[RIGHT_ELBOW, :2] = right_elbow

    knee_shift = 0.012 * math.sin(math.pi * t)
    lm[LEFT_KNEE, :2] = [0.455 + sway - knee_shift, 0.78]
    lm[RIGHT_KNEE, :2] = [0.555 + sway + knee_shift, 0.78]
    lm[LEFT_ANKLE, :2] = [0.44, 0.94]
    lm[RIGHT_ANKLE, :2] = [0.57, 0.94]
    lm[LEFT_HEEL, :2] = [0.425, 0.955]
    lm[RIGHT_HEEL, :2] = [0.585, 0.955]
    lm[LEFT_FOOT, :2] = [0.40, 0.96]
    lm[RIGHT_FOOT, :2] = [0.61, 0.96]

    # Small synthetic depth values, useful for future 3D extensions.
    lm[:, 2] = 0.02 * np.sin(2 * np.pi * t)
    return lm


def create_demo_video(output_path: Path, seconds: float = 6.0,
                      fps: float = 30.0, width: int = 720, height: int = 540) -> Tuple[Path, List[Dict[str, Any]]]:
    total_frames = int(seconds * fps)
    writer = cv2.VideoWriter(
        str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
    )
    records: List[Dict[str, Any]] = []
    for frame_index in range(total_frames):
        background = np.full((height, width, 3), 238, dtype=np.uint8)
        cv2.line(background, (0, int(height * 0.96)), (width, int(height * 0.96)), (110, 110, 110), 2)
        landmarks = demo_landmarks(frame_index, total_frames)
        frame = draw_pose(background, landmarks)
        hand = midpoint(point(landmarks, LEFT_WRIST), point(landmarks, RIGHT_WRIST))
        club_end = hand + np.array([0.08, 0.14])
        cv2.line(
            frame,
            (int(hand[0] * width), int(hand[1] * height)),
            (int(club_end[0] * width), int(club_end[1] * height)),
            (50, 50, 50), 3, cv2.LINE_AA
        )
        cv2.putText(frame, "Synthetic demo - replace with a real upload", (18, height - 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (30, 30, 30), 2, cv2.LINE_AA)
        writer.write(frame)
        if frame_index % FRAME_STRIDE == 0:
            records.append({
                "frame_index": frame_index,
                "timestamp_s": frame_index / fps,
                "landmarks": landmarks,
                "detected": True,
                "source": "synthetic_demo"
            })
    writer.release()
    return output_path, records

## 4. Select or upload the input video

The default demo is intentionally synthetic. It verifies that every downstream cell runs before a real data collection process is established.

In [ ]:
INPUT_VIDEO_PATH: Path
DEMO_RECORDS: Optional[List[Dict[str, Any]]] = None

if VIDEO_MODE.lower() == "demo":
    INPUT_VIDEO_PATH, DEMO_RECORDS = create_demo_video(PROJECT_DIR / "demo_golf_swing.mp4")
    print("Created synthetic demo video:", INPUT_VIDEO_PATH)

elif VIDEO_MODE.lower() == "upload":
    if not IN_COLAB:
        raise RuntimeError("Upload mode requires Google Colab. Use VIDEO_MODE='path' outside Colab.")
    print("Upload one golf video now.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No video was uploaded.")
    uploaded_name = next(iter(uploaded))
    raw_path = PROJECT_DIR / Path(uploaded_name).name
    raw_path.write_bytes(uploaded[uploaded_name])
    INPUT_VIDEO_PATH = standardize_video(raw_path, PROJECT_DIR / "input_standardized.mp4", MAX_OUTPUT_WIDTH)
    print("Prepared input video:", INPUT_VIDEO_PATH)

elif VIDEO_MODE.lower() == "path":
    raw_path = Path(LOCAL_VIDEO_PATH).expanduser().resolve()
    if not raw_path.exists():
        raise FileNotFoundError(f"Video path not found: {raw_path}")
    INPUT_VIDEO_PATH = standardize_video(raw_path, PROJECT_DIR / "input_standardized.mp4", MAX_OUTPUT_WIDTH)
    print("Prepared input video:", INPUT_VIDEO_PATH)

else:
    raise ValueError("VIDEO_MODE must be 'demo', 'upload', or 'path'.")

display(Video(str(INPUT_VIDEO_PATH), embed=True, width=720))

## 5. Pose extraction and frame-level biomechanics features

For a real upload, the notebook runs MediaPipe in `VIDEO` mode and supplies monotonically increasing frame timestamps. The analysis is two-dimensional and camera-dependent. A future production system should combine multiple camera views or calibrated 3D capture.

In [ ]:
def landmarks_to_array(pose_landmarks: Sequence[Any]) -> np.ndarray:
    array = np.zeros((33, 5), dtype=np.float32)
    for i, landmark in enumerate(pose_landmarks[:33]):
        array[i] = [
            float(landmark.x), float(landmark.y), float(getattr(landmark, "z", 0.0)),
            float(getattr(landmark, "visibility", 1.0)),
            float(getattr(landmark, "presence", 1.0)),
        ]
    return array


def extract_frame_features(landmarks: np.ndarray, timestamp_s: float, frame_index: int) -> Dict[str, float]:
    ls, rs = point(landmarks, LEFT_SHOULDER), point(landmarks, RIGHT_SHOULDER)
    le, re = point(landmarks, LEFT_ELBOW), point(landmarks, RIGHT_ELBOW)
    lw, rw = point(landmarks, LEFT_WRIST), point(landmarks, RIGHT_WRIST)
    lh, rh = point(landmarks, LEFT_HIP), point(landmarks, RIGHT_HIP)
    lk, rk = point(landmarks, LEFT_KNEE), point(landmarks, RIGHT_KNEE)
    la, ra = point(landmarks, LEFT_ANKLE), point(landmarks, RIGHT_ANKLE)
    nose = point(landmarks, NOSE)

    shoulder_center = midpoint(ls, rs)
    hip_center = midpoint(lh, rh)
    hand_center = midpoint(lw, rw)
    shoulder_width = max(euclidean(ls, rs), 1e-4)
    hip_width = max(euclidean(lh, rh), 1e-4)

    torso_vector = shoulder_center - hip_center
    torso_tilt = abs(float(np.degrees(np.arctan2(torso_vector[0], -torso_vector[1]))))
    shoulder_line_angle = line_angle_degrees(ls, rs)
    hip_line_angle = line_angle_degrees(lh, rh)

    return {
        "frame_index": int(frame_index),
        "timestamp_s": float(timestamp_s),
        "left_elbow_angle": angle_three_points(ls, le, lw),
        "right_elbow_angle": angle_three_points(rs, re, rw),
        "left_knee_angle": angle_three_points(lh, lk, la),
        "right_knee_angle": angle_three_points(rh, rk, ra),
        "left_hip_angle": angle_three_points(ls, lh, lk),
        "right_hip_angle": angle_three_points(rs, rh, rk),
        "torso_tilt_deg": torso_tilt,
        "shoulder_line_angle_deg": shoulder_line_angle,
        "hip_line_angle_deg": hip_line_angle,
        "shoulder_hip_separation_deg": wrapped_angle_difference(shoulder_line_angle, hip_line_angle),
        "shoulder_width": shoulder_width,
        "hip_width": hip_width,
        "shoulder_center_x": float(shoulder_center[0]),
        "shoulder_center_y": float(shoulder_center[1]),
        "hip_center_x": float(hip_center[0]),
        "hip_center_y": float(hip_center[1]),
        "hand_center_x": float(hand_center[0]),
        "hand_center_y": float(hand_center[1]),
        "nose_x": float(nose[0]),
        "nose_y": float(nose[1]),
        "mean_visibility": float(np.nanmean(landmarks[:, 3])),
    }


def analyze_real_video(video_path: Path) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    if mp is None:
        raise RuntimeError("MediaPipe is unavailable. Rerun the install/import cells.")
    model_path = download_pose_model()
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"OpenCV could not open {video_path}")

    fps = float(capture.get(cv2.CAP_PROP_FPS) or 30.0)
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    max_frames = min(total_frames if total_frames > 0 else int(MAX_SECONDS * fps), int(MAX_SECONDS * fps))

    BaseOptions = mp.tasks.BaseOptions
    PoseLandmarker = mp.tasks.vision.PoseLandmarker
    PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(model_path)),
        running_mode=VisionRunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.45,
        min_pose_presence_confidence=0.45,
        min_tracking_confidence=0.45,
        output_segmentation_masks=False,
    )

    records: List[Dict[str, Any]] = []
    analyzed = 0
    detected = 0
    frame_index = 0
    started = time.time()
    with PoseLandmarker.create_from_options(options) as landmarker:
        while frame_index < max_frames:
            ok, frame = capture.read()
            if not ok:
                break
            if frame_index % FRAME_STRIDE == 0:
                analyzed += 1
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                timestamp_ms = int(round(frame_index * 1000.0 / fps))
                result = landmarker.detect_for_video(mp_image, timestamp_ms)
                if result.pose_landmarks:
                    landmarks = landmarks_to_array(result.pose_landmarks[0])
                    detected += 1
                    records.append({
                        "frame_index": frame_index,
                        "timestamp_s": frame_index / fps,
                        "landmarks": landmarks,
                        "detected": True,
                        "source": "mediapipe"
                    })
                else:
                    records.append({
                        "frame_index": frame_index,
                        "timestamp_s": frame_index / fps,
                        "landmarks": None,
                        "detected": False,
                        "source": "mediapipe"
                    })
            frame_index += 1
    capture.release()

    metadata = {
        "fps": fps,
        "total_frames_reported": total_frames,
        "frames_processed_limit": frame_index,
        "width": width,
        "height": height,
        "samples_analyzed": analyzed,
        "samples_detected": detected,
        "detection_coverage": detected / max(analyzed, 1),
        "analysis_seconds": time.time() - started,
    }
    return records, metadata


def metadata_from_video(video_path: Path, records: List[Dict[str, Any]]) -> Dict[str, Any]:
    capture = cv2.VideoCapture(str(video_path))
    fps = float(capture.get(cv2.CAP_PROP_FPS) or 30.0)
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    capture.release()
    detected = sum(bool(record["detected"]) for record in records)
    return {
        "fps": fps,
        "total_frames_reported": total_frames,
        "frames_processed_limit": total_frames,
        "width": width,
        "height": height,
        "samples_analyzed": len(records),
        "samples_detected": detected,
        "detection_coverage": detected / max(len(records), 1),
        "analysis_seconds": 0.0,
    }


if VIDEO_MODE.lower() == "demo":
    pose_records = copy.deepcopy(DEMO_RECORDS or [])
    video_metadata = metadata_from_video(INPUT_VIDEO_PATH, pose_records)
else:
    pose_records, video_metadata = analyze_real_video(INPUT_VIDEO_PATH)

for record in pose_records:
    if record["detected"] and record["landmarks"] is not None:
        record["features"] = extract_frame_features(
            record["landmarks"], record["timestamp_s"], record["frame_index"]
        )
    else:
        record["features"] = None

print(json.dumps(video_metadata, indent=2))
if video_metadata["detection_coverage"] < 0.60:
    print("WARNING: Pose coverage is low. Use a full-body view, stable camera, and stronger lighting.")

## 6. Temporal features, swing phases, and heuristic score

The phase detector is a transparent heuristic based on hand height and hand speed. It is useful for an MVP, but a labeled phase-classification dataset is required for production.

In [ ]:
def build_feature_dataframe(records: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = [record["features"] for record in records if record.get("features") is not None]
    if not rows:
        raise RuntimeError("No body pose was detected in any analyzed frame.")
    df = pd.DataFrame(rows).sort_values("frame_index").reset_index(drop=True)
    numeric_cols = [column for column in df.columns if column not in {"frame_index"}]
    df[numeric_cols] = df[numeric_cols].interpolate(limit_direction="both")

    dt = df["timestamp_s"].diff().replace(0, np.nan)
    scale = df["shoulder_width"].rolling(5, min_periods=1, center=True).median().clip(lower=1e-4)
    hand_dx = df["hand_center_x"].diff()
    hand_dy = df["hand_center_y"].diff()
    raw_hand_speed = np.sqrt(hand_dx ** 2 + hand_dy ** 2) / dt / scale
    df["hand_speed_shoulder_widths_per_s"] = (
        raw_hand_speed.replace([np.inf, -np.inf], np.nan)
        .interpolate(limit_direction="both")
        .rolling(5, min_periods=1, center=True).mean()
    )

    initial_count = max(3, min(10, len(df) // 8 if len(df) >= 8 else len(df)))
    initial_hip_x = float(df["hip_center_x"].iloc[:initial_count].median())
    initial_nose_x = float(df["nose_x"].iloc[:initial_count].median())
    df["hip_sway_shoulder_widths"] = (df["hip_center_x"] - initial_hip_x).abs() / scale
    df["head_motion_shoulder_widths"] = (df["nose_x"] - initial_nose_x).abs() / scale

    speed_dt = df["timestamp_s"].diff().replace(0, np.nan)
    df["hand_acceleration"] = (
        df["hand_speed_shoulder_widths_per_s"].diff() / speed_dt
    ).replace([np.inf, -np.inf], np.nan).interpolate(limit_direction="both")
    return df


def assign_phases(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, int]]:
    output = df.copy()
    if len(output) < 8:
        output["phase"] = "Unsegmented"
        return output, {"start": 0, "top": max(0, len(output) // 2), "impact": max(0, len(output) - 2)}

    smoothed_hand_y = output["hand_center_y"].rolling(7, min_periods=1, center=True).mean()
    search_start = max(1, int(0.10 * len(output)))
    search_end = max(search_start + 1, int(0.75 * len(output)))
    top_pos = int(smoothed_hand_y.iloc[search_start:search_end].idxmin())

    post_top_end = max(top_pos + 2, int(0.90 * len(output)))
    speed_slice = output.loc[top_pos:post_top_end, "hand_speed_shoulder_widths_per_s"]
    impact_pos = int(speed_slice.idxmax()) if len(speed_slice) else min(top_pos + 1, len(output) - 1)
    impact_pos = max(top_pos + 1, min(impact_pos, len(output) - 2))
    address_end = max(1, int(top_pos * 0.18))

    phases = []
    for i in range(len(output)):
        if i <= address_end:
            phase = "Address"
        elif i < top_pos:
            phase = "Backswing"
        elif i == top_pos:
            phase = "Top"
        elif i < impact_pos:
            phase = "Downswing"
        elif i == impact_pos:
            phase = "Impact"
        else:
            phase = "Follow-through"
        phases.append(phase)
    output["phase"] = phases
    return output, {"start": 0, "address_end": address_end, "top": top_pos, "impact": impact_pos}


def band_score(value: float, ideal_low: float, ideal_high: float,
               outer_low: float, outer_high: float) -> float:
    if not np.isfinite(value):
        return 0.0
    if ideal_low <= value <= ideal_high:
        return 100.0
    if value < ideal_low:
        return 100.0 * clamp((value - outer_low) / max(ideal_low - outer_low, 1e-6), 0.0, 1.0)
    return 100.0 * clamp((outer_high - value) / max(outer_high - ideal_high, 1e-6), 0.0, 1.0)


def upper_bound_score(value: float, ideal_max: float, outer_max: float) -> float:
    if not np.isfinite(value):
        return 0.0
    if value <= ideal_max:
        return 100.0
    return 100.0 * clamp((outer_max - value) / max(outer_max - ideal_max, 1e-6), 0.0, 1.0)


def summarize_swing(df: pd.DataFrame, phase_indices: Dict[str, int],
                    detection_coverage: float) -> Tuple[Dict[str, float], Dict[str, float]]:
    address = df[df["phase"] == "Address"]
    impact = df.iloc[[phase_indices["impact"]]]
    top = df.iloc[[phase_indices["top"]]]

    start_t = float(df["timestamp_s"].iloc[0])
    top_t = float(top["timestamp_s"].iloc[0])
    impact_t = float(impact["timestamp_s"].iloc[0])
    backswing_duration = max(top_t - start_t, 1e-3)
    downswing_duration = max(impact_t - top_t, 1e-3)
    tempo_ratio = backswing_duration / downswing_duration

    acceleration_std = float(df["hand_acceleration"].replace([np.inf, -np.inf], np.nan).dropna().std() or 0.0)
    smoothness_index = 1.0 / (1.0 + max(acceleration_std, 0.0))

    summary = {
        "detection_coverage": float(detection_coverage),
        "duration_s": float(df["timestamp_s"].iloc[-1] - df["timestamp_s"].iloc[0]),
        "address_torso_tilt_deg": float(address["torso_tilt_deg"].median()),
        "address_left_knee_angle_deg": float(address["left_knee_angle"].median()),
        "address_right_knee_angle_deg": float(address["right_knee_angle"].median()),
        "max_shoulder_hip_separation_deg": float(df["shoulder_hip_separation_deg"].max()),
        "max_hip_sway_shoulder_widths": float(df["hip_sway_shoulder_widths"].max()),
        "max_head_motion_shoulder_widths": float(df["head_motion_shoulder_widths"].max()),
        "impact_left_elbow_angle_deg": float(impact["left_elbow_angle"].iloc[0]),
        "impact_right_elbow_angle_deg": float(impact["right_elbow_angle"].iloc[0]),
        "peak_hand_speed_shoulder_widths_per_s": float(df["hand_speed_shoulder_widths_per_s"].max()),
        "backswing_duration_s": backswing_duration,
        "downswing_duration_s": downswing_duration,
        "tempo_ratio": float(tempo_ratio),
        "smoothness_index": float(smoothness_index),
    }
    summary["address_mean_knee_angle_deg"] = (
        summary["address_left_knee_angle_deg"] + summary["address_right_knee_angle_deg"]
    ) / 2.0
    summary["impact_mean_elbow_angle_deg"] = (
        summary["impact_left_elbow_angle_deg"] + summary["impact_right_elbow_angle_deg"]
    ) / 2.0

    components = {
        # Wide bands are intentional because camera perspective changes 2D measurements.
        "Posture": 0.55 * band_score(summary["address_torso_tilt_deg"], 0, 28, 0, 60)
                   + 0.45 * band_score(summary["address_mean_knee_angle_deg"], 138, 178, 105, 180),
        "Balance": 0.60 * upper_bound_score(summary["max_hip_sway_shoulder_widths"], 0.18, 0.65)
                   + 0.40 * upper_bound_score(summary["max_head_motion_shoulder_widths"], 0.22, 0.75),
        "Rotation": band_score(summary["max_shoulder_hip_separation_deg"], 8, 45, 0, 85),
        "Tempo": band_score(summary["tempo_ratio"], 1.7, 4.0, 0.7, 7.0),
        "Arm extension": band_score(summary["impact_mean_elbow_angle_deg"], 128, 180, 85, 180),
        "Consistency": 100.0 * clamp(summary["smoothness_index"] / 0.20, 0.0, 1.0),
        "Pose confidence": 100.0 * clamp(summary["detection_coverage"], 0.0, 1.0),
    }
    weights = {
        "Posture": 0.18, "Balance": 0.20, "Rotation": 0.17,
        "Tempo": 0.15, "Arm extension": 0.12, "Consistency": 0.10,
        "Pose confidence": 0.08,
    }
    components = {key: float(clamp(value, 0, 100)) for key, value in components.items()}
    overall = sum(components[key] * weights[key] for key in weights)
    summary["heuristic_overall_score"] = float(clamp(overall, 0, 100))
    return summary, components


feature_df = build_feature_dataframe(pose_records)
feature_df, phase_indices = assign_phases(feature_df)
swing_summary, component_scores = summarize_swing(
    feature_df, phase_indices, video_metadata["detection_coverage"]
)

phase_by_frame = dict(zip(feature_df["frame_index"].astype(int), feature_df["phase"]))
for record in pose_records:
    if record["features"] is not None:
        record["phase"] = phase_by_frame.get(record["frame_index"], "Unsegmented")
    else:
        record["phase"] = "Pose not detected"

print("Prototype overall score:", round(swing_summary["heuristic_overall_score"], 1))
display(pd.DataFrame(component_scores.items(), columns=["Component", "Score"]).sort_values("Score"))
display(pd.DataFrame([swing_summary]).T.rename(columns={0: "Value"}))

## 7. Annotated output video

The output overlays the skeleton, phase, score, elbow angles, and detected hand speed. For uploaded videos, the notebook attempts to preserve the original audio while converting the final result to browser-compatible H.264.

In [ ]:
def create_annotated_video(input_path: Path, output_path: Path,
                           records: List[Dict[str, Any]], df: pd.DataFrame,
                           overall_score: float) -> Path:
    capture = cv2.VideoCapture(str(input_path))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open video: {input_path}")
    fps = float(capture.get(cv2.CAP_PROP_FPS) or 30.0)
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 640)
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 480)
    total_limit = min(
        int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or int(MAX_SECONDS * fps)),
        int(MAX_SECONDS * fps)
    )

    temp_path = output_path.with_name("annotated_temp_mp4v.mp4")
    writer = cv2.VideoWriter(
        str(temp_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
    )
    record_map = {int(record["frame_index"]): record for record in records}
    df_map = {int(row.frame_index): row for row in df.itertuples(index=False)}
    sorted_sample_frames = sorted(record_map)
    sample_pointer = 0
    current_record = None
    current_row = None

    frame_index = 0
    while frame_index < total_limit:
        ok, frame = capture.read()
        if not ok:
            break
        while sample_pointer < len(sorted_sample_frames) and sorted_sample_frames[sample_pointer] <= frame_index:
            sample_frame = sorted_sample_frames[sample_pointer]
            candidate = record_map[sample_frame]
            if candidate.get("landmarks") is not None:
                current_record = candidate
                current_row = df_map.get(sample_frame, current_row)
            sample_pointer += 1

        landmarks = current_record.get("landmarks") if current_record else None
        phase = current_record.get("phase", "No pose") if current_record else "No pose"
        extra = []
        if current_row is not None:
            extra = [
                f"Elbows L/R: {current_row.left_elbow_angle:.0f}/{current_row.right_elbow_angle:.0f} deg",
                f"Hand speed: {current_row.hand_speed_shoulder_widths_per_s:.2f} shoulder-widths/s",
            ]
        annotated = draw_pose(frame, landmarks, phase, overall_score, extra)
        writer.write(annotated)
        frame_index += 1

    capture.release()
    writer.release()

    # Use H.264 for broad browser compatibility and copy original audio when available.
    command_with_audio = [
        "ffmpeg", "-y", "-i", str(temp_path), "-i", str(input_path),
        "-map", "0:v:0", "-map", "1:a?", "-c:v", "libx264", "-preset", "veryfast",
        "-crf", "23", "-pix_fmt", "yuv420p", "-c:a", "aac", "-shortest",
        "-movflags", "+faststart", str(output_path)
    ]
    if not run_ffmpeg(command_with_audio):
        shutil.copy2(temp_path, output_path)
    temp_path.unlink(missing_ok=True)
    return output_path


ANNOTATED_VIDEO_PATH = create_annotated_video(
    INPUT_VIDEO_PATH,
    OUTPUT_DIR / "isaiah_goh_golf_swing_annotated.mp4",
    pose_records,
    feature_df,
    swing_summary["heuristic_overall_score"],
)
print("Annotated video:", ANNOTATED_VIDEO_PATH)
display(Video(str(ANNOTATED_VIDEO_PATH), embed=True, width=720))

## 8. Diagnostic visualizations

These plots expose the measured signals instead of hiding the scoring process. Large jumps often indicate occlusion, camera motion, or landmark tracking errors rather than a real biomechanics change.

In [ ]:
PLOT_PATHS: Dict[str, Path] = {}

plt.figure(figsize=(12, 5))
plt.plot(feature_df["timestamp_s"], feature_df["left_elbow_angle"], label="Left elbow")
plt.plot(feature_df["timestamp_s"], feature_df["right_elbow_angle"], label="Right elbow")
plt.plot(feature_df["timestamp_s"], feature_df["left_knee_angle"], label="Left knee")
plt.plot(feature_df["timestamp_s"], feature_df["right_knee_angle"], label="Right knee")
plt.xlabel("Time (seconds)")
plt.ylabel("Angle (degrees)")
plt.title("Joint Angles Across the Swing")
plt.legend(ncol=2)
plt.grid(alpha=0.25)
plt.tight_layout()
PLOT_PATHS["joint_angles"] = OUTPUT_DIR / "joint_angles.png"
plt.savefig(PLOT_PATHS["joint_angles"], dpi=160)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(feature_df["timestamp_s"], feature_df["hand_speed_shoulder_widths_per_s"], label="Hand speed")
plt.plot(feature_df["timestamp_s"], feature_df["hip_sway_shoulder_widths"], label="Hip sway")
plt.plot(feature_df["timestamp_s"], feature_df["head_motion_shoulder_widths"], label="Head motion")
plt.xlabel("Time (seconds)")
plt.ylabel("Normalized value")
plt.title("Tempo, Balance, and Stability Indicators")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
PLOT_PATHS["stability"] = OUTPUT_DIR / "stability_and_speed.png"
plt.savefig(PLOT_PATHS["stability"], dpi=160)
plt.show()

component_df = pd.DataFrame(component_scores.items(), columns=["Component", "Score"]).sort_values("Score")
plt.figure(figsize=(9, 5))
plt.barh(component_df["Component"], component_df["Score"])
plt.xlim(0, 100)
plt.xlabel("Prototype score")
plt.title("Golf Coach Component Scores")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
PLOT_PATHS["components"] = OUTPUT_DIR / "component_scores.png"
plt.savefig(PLOT_PATHS["components"], dpi=160)
plt.show()

feature_df.to_csv(OUTPUT_DIR / "swing_frame_features.csv", index=False)

## 9. Supervised machine learning - Random Forest

A Random Forest demonstrates the supervised-learning architecture. Because this notebook does not ship a licensed, coach-labeled golf dataset, it generates synthetic feature rows and targets from transparent scoring rules. Replace this block with real labeled swings before claiming predictive validity.

A future labeled CSV should contain the feature columns below and a `coach_score` target from qualified coaches. It should also include camera view, club type, shot type, handedness, and repeated labels to estimate inter-rater agreement.

In [ ]:
ML_FEATURE_COLUMNS = [
    "address_torso_tilt_deg",
    "address_mean_knee_angle_deg",
    "max_shoulder_hip_separation_deg",
    "max_hip_sway_shoulder_widths",
    "max_head_motion_shoulder_widths",
    "impact_mean_elbow_angle_deg",
    "tempo_ratio",
    "smoothness_index",
    "detection_coverage",
]


def transparent_target(row: Dict[str, float]) -> float:
    posture = 0.55 * band_score(row["address_torso_tilt_deg"], 0, 28, 0, 60) + \
              0.45 * band_score(row["address_mean_knee_angle_deg"], 138, 178, 105, 180)
    balance = 0.60 * upper_bound_score(row["max_hip_sway_shoulder_widths"], 0.18, 0.65) + \
              0.40 * upper_bound_score(row["max_head_motion_shoulder_widths"], 0.22, 0.75)
    rotation = band_score(row["max_shoulder_hip_separation_deg"], 8, 45, 0, 85)
    tempo = band_score(row["tempo_ratio"], 1.7, 4.0, 0.7, 7.0)
    extension = band_score(row["impact_mean_elbow_angle_deg"], 128, 180, 85, 180)
    consistency = 100.0 * clamp(row["smoothness_index"] / 0.20, 0, 1)
    confidence = 100.0 * clamp(row["detection_coverage"], 0, 1)
    return float(clamp(
        0.18 * posture + 0.20 * balance + 0.17 * rotation + 0.15 * tempo +
        0.12 * extension + 0.10 * consistency + 0.08 * confidence,
        0, 100
    ))


def generate_synthetic_labeled_data(n_samples: int = 1800, seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    data = pd.DataFrame({
        "address_torso_tilt_deg": rng.uniform(0, 65, n_samples),
        "address_mean_knee_angle_deg": rng.uniform(100, 180, n_samples),
        "max_shoulder_hip_separation_deg": rng.uniform(0, 90, n_samples),
        "max_hip_sway_shoulder_widths": rng.uniform(0, 0.85, n_samples),
        "max_head_motion_shoulder_widths": rng.uniform(0, 0.95, n_samples),
        "impact_mean_elbow_angle_deg": rng.uniform(75, 180, n_samples),
        "tempo_ratio": rng.uniform(0.5, 7.5, n_samples),
        "smoothness_index": rng.uniform(0.02, 0.45, n_samples),
        "detection_coverage": rng.uniform(0.45, 1.0, n_samples),
    })
    scores = [transparent_target(row) for row in data.to_dict(orient="records")]
    data["coach_score"] = np.clip(np.asarray(scores) + rng.normal(0, 3.0, n_samples), 0, 100)
    data["training_source"] = "synthetic_rule_based"
    return data


labeled_df = generate_synthetic_labeled_data()
X = labeled_df[ML_FEATURE_COLUMNS]
y = labeled_df["coach_score"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.22, random_state=SEED
)

rf_model = RandomForestRegressor(
    n_estimators=260,
    max_depth=12,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=SEED,
)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_r2 = r2_score(y_test, rf_predictions)

current_ml_row = pd.DataFrame([{column: swing_summary[column] for column in ML_FEATURE_COLUMNS}])
rf_swing_score = float(rf_model.predict(current_ml_row)[0])
swing_summary["random_forest_score"] = rf_swing_score

joblib.dump(rf_model, MODEL_DIR / "random_forest_swing_score.joblib")
labeled_df.to_csv(OUTPUT_DIR / "synthetic_ml_training_data.csv", index=False)

importance_df = pd.DataFrame({
    "Feature": ML_FEATURE_COLUMNS,
    "Importance": rf_model.feature_importances_,
}).sort_values("Importance", ascending=True)

print(f"Synthetic holdout MAE: {rf_mae:.2f} points")
print(f"Synthetic holdout R-squared: {rf_r2:.3f}")
print(f"Current swing Random Forest score: {rf_swing_score:.1f}/100")
print("These validation metrics measure recovery of the synthetic rule system, not real coaching accuracy.")

plt.figure(figsize=(9, 5))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.xlabel("Random Forest importance")
plt.title("Supervised ML Feature Importance - Synthetic Demonstration")
plt.tight_layout()
PLOT_PATHS["ml_importance"] = OUTPUT_DIR / "random_forest_feature_importance.png"
plt.savefig(PLOT_PATHS["ml_importance"], dpi=160)
plt.show()

### Optional replacement with real labeled data

Set `CUSTOM_LABELED_CSV` to a file path and rerun this cell after preparing a de-identified dataset. The code validates the required columns and trains a new model.

In [ ]:
CUSTOM_LABELED_CSV = ""  # Example: "/content/coach_labeled_swings.csv"

if CUSTOM_LABELED_CSV:
    custom_path = Path(CUSTOM_LABELED_CSV)
    real_labeled_df = pd.read_csv(custom_path)
    required = set(ML_FEATURE_COLUMNS + ["coach_score"])
    missing = required.difference(real_labeled_df.columns)
    if missing:
        raise ValueError(f"The labeled CSV is missing: {sorted(missing)}")
    real_labeled_df = real_labeled_df.dropna(subset=list(required)).copy()
    X_real = real_labeled_df[ML_FEATURE_COLUMNS]
    y_real = real_labeled_df["coach_score"].clip(0, 100)
    X_train, X_test, y_train, y_test = train_test_split(
        X_real, y_real, test_size=0.22, random_state=SEED
    )
    real_rf_model = RandomForestRegressor(
        n_estimators=350, max_depth=14, min_samples_leaf=3,
        n_jobs=-1, random_state=SEED
    )
    real_rf_model.fit(X_train, y_train)
    pred = real_rf_model.predict(X_test)
    print("Real-data MAE:", mean_absolute_error(y_test, pred))
    print("Real-data R-squared:", r2_score(y_test, pred))
    joblib.dump(real_rf_model, MODEL_DIR / "random_forest_real_labeled.joblib")

## 10. Deep learning - GRU sequence model

A GRU processes the swing as an ordered time series, unlike the summary-based Random Forest. This cell augments the current sequence with controlled noise to demonstrate training. It is an architectural proof of concept, not an independently validated model.

In [ ]:
DL_SEQUENCE_FEATURES = [
    "torso_tilt_deg",
    "shoulder_hip_separation_deg",
    "left_elbow_angle",
    "right_elbow_angle",
    "left_knee_angle",
    "right_knee_angle",
    "hip_sway_shoulder_widths",
    "head_motion_shoulder_widths",
    "hand_center_y",
    "hand_speed_shoulder_widths_per_s",
]
SEQUENCE_LENGTH = 60


def resample_sequence(df: pd.DataFrame, columns: Sequence[str], length: int = 60) -> np.ndarray:
    values = df[list(columns)].replace([np.inf, -np.inf], np.nan).interpolate(limit_direction="both")
    values = values.fillna(values.median(numeric_only=True)).fillna(0.0).to_numpy(dtype=np.float32)
    old_x = np.linspace(0.0, 1.0, len(values))
    new_x = np.linspace(0.0, 1.0, length)
    resampled = np.column_stack([
        np.interp(new_x, old_x, values[:, column_index])
        for column_index in range(values.shape[1])
    ])
    return resampled.astype(np.float32)


def make_synthetic_sequences(base_sequence: np.ndarray, base_score: float,
                             n_sequences: int = 420, seed: int = SEED) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    sequences = []
    targets = []
    scale = np.std(base_sequence, axis=0, keepdims=True)
    scale = np.where(scale < 1e-3, 1.0, scale)
    for _ in range(n_sequences):
        noise_level = float(rng.uniform(0.015, 0.45))
        drift_level = float(rng.uniform(-0.18, 0.18))
        noise = rng.normal(0, noise_level, base_sequence.shape).astype(np.float32) * scale
        drift = np.linspace(0, drift_level, base_sequence.shape[0], dtype=np.float32)[:, None] * scale
        sequence = base_sequence + noise + drift
        quality_penalty = 52.0 * noise_level + 24.0 * abs(drift_level)
        target = clamp(base_score - quality_penalty + rng.normal(0, 2.0), 0, 100)
        sequences.append(sequence)
        targets.append(target)
    sequences.append(base_sequence.copy())
    targets.append(float(base_score))
    return np.asarray(sequences, dtype=np.float32), np.asarray(targets, dtype=np.float32)


class SwingGRU(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 48):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=0.15,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        sequence_output, _ = self.gru(x)
        final_state = sequence_output[:, -1, :]
        return self.head(final_state).squeeze(-1) * 100.0


if RUN_DEEP_LEARNING:
    base_sequence = resample_sequence(feature_df, DL_SEQUENCE_FEATURES, SEQUENCE_LENGTH)
    X_seq, y_seq = make_synthetic_sequences(
        base_sequence, swing_summary["heuristic_overall_score"]
    )
    indices = np.arange(len(X_seq))
    train_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=SEED)

    train_mean = X_seq[train_idx].mean(axis=(0, 1), keepdims=True)
    train_std = X_seq[train_idx].std(axis=(0, 1), keepdims=True)
    train_std = np.where(train_std < 1e-5, 1.0, train_std)
    X_scaled = (X_seq - train_mean) / train_std

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SwingGRU(input_size=len(DL_SEQUENCE_FEATURES)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=1e-4)
    loss_function = nn.SmoothL1Loss()

    train_X = torch.tensor(X_scaled[train_idx], dtype=torch.float32)
    train_y = torch.tensor(y_seq[train_idx], dtype=torch.float32)
    batch_size = 48
    history = []

    model.train()
    for epoch in range(DL_EPOCHS):
        permutation = torch.randperm(len(train_X))
        epoch_losses = []
        for start in range(0, len(train_X), batch_size):
            batch_indices = permutation[start:start + batch_size]
            batch_X = train_X[batch_indices].to(device)
            batch_y = train_y[batch_indices].to(device)
            optimizer.zero_grad(set_to_none=True)
            prediction = model(batch_X)
            loss = loss_function(prediction, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu()))
        history.append(np.mean(epoch_losses))

    model.eval()
    with torch.no_grad():
        test_tensor = torch.tensor(X_scaled[test_idx], dtype=torch.float32).to(device)
        test_predictions = model(test_tensor).cpu().numpy()
        actual_scaled = ((base_sequence[None, ...] - train_mean) / train_std).astype(np.float32)
        actual_prediction = float(model(torch.tensor(actual_scaled).to(device)).cpu().item())

    dl_mae = mean_absolute_error(y_seq[test_idx], test_predictions)
    swing_summary["gru_sequence_score"] = actual_prediction

    torch.save({
        "model_state_dict": model.state_dict(),
        "input_features": DL_SEQUENCE_FEATURES,
        "sequence_length": SEQUENCE_LENGTH,
        "train_mean": train_mean,
        "train_std": train_std,
    }, MODEL_DIR / "swing_gru_demo.pt")

    print("Device:", device)
    print(f"Synthetic sequence holdout MAE: {dl_mae:.2f} points")
    print(f"Current swing GRU score: {actual_prediction:.1f}/100")
    print("This model is trained on perturbations of the analyzed sequence; it is a pipeline demonstration.")

    plt.figure(figsize=(8, 4))
    plt.plot(np.arange(1, len(history) + 1), history)
    plt.xlabel("Epoch")
    plt.ylabel("Smooth L1 loss")
    plt.title("GRU Training Loss - Synthetic Sequence Demonstration")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    PLOT_PATHS["gru_loss"] = OUTPUT_DIR / "gru_training_loss.png"
    plt.savefig(PLOT_PATHS["gru_loss"], dpi=160)
    plt.show()
else:
    swing_summary["gru_sequence_score"] = float("nan")
    print("Deep learning skipped because RUN_DEEP_LEARNING is False.")

## 11. Coaching advice and personalized four-week plan

Advice is selected from the lowest-scoring components and linked to observable metrics. Drills are general golf-practice suggestions. Stop any drill that causes pain, dizziness, or unusual shortness of breath, and seek appropriate professional guidance.

In [ ]:
ADVICE_LIBRARY: Dict[str, Dict[str, str]] = {
    "Posture": {
        "insight": "Your address posture or knee-flex indicator is outside the prototype reference band.",
        "tip": "Build a repeatable address: feet stable, knees softly flexed, spine long, and pressure centered over the mid-foot.",
        "drill": "Mirror setup drill: hold the address position for 10 seconds, reset, and repeat 8 times before hitting balls.",
    },
    "Balance": {
        "insight": "The pose track shows more hip sway or head motion than the prototype target.",
        "tip": "Keep rotational movement centered rather than sliding laterally through the swing.",
        "drill": "Feet-together half swings: make 3 sets of 8 controlled swings at 50-60% effort.",
    },
    "Rotation": {
        "insight": "Shoulder-to-hip separation is limited or excessive in the two-dimensional view.",
        "tip": "Coordinate chest and pelvis rotation without forcing range of motion.",
        "drill": "Club-across-chest turns: 2 sets of 10 slow turns, then 10 half swings with the same sequence.",
    },
    "Tempo": {
        "insight": "The backswing-to-downswing timing ratio is outside the broad prototype band.",
        "tip": "Use a calm backswing and a continuous transition rather than rushing from the top.",
        "drill": "Count 'one-two-three' to the top and 'one' through impact for 20 rehearsal swings.",
    },
    "Arm extension": {
        "insight": "The impact-frame elbow-extension indicator may show early collapse or an occluded arm.",
        "tip": "Allow the arms to extend through the strike while the torso continues rotating.",
        "drill": "Towel-target extension drill: place a towel about one clubhead past the ball line and make 15 slow swings toward it.",
    },
    "Consistency": {
        "insight": "The detected hand-speed curve contains abrupt changes.",
        "tip": "Prioritize repeatable motion before adding speed.",
        "drill": "Three-speed ladder: 5 swings at 40%, 5 at 55%, and 5 at 70%, keeping the same finish balance.",
    },
    "Pose confidence": {
        "insight": "Landmark detection coverage is not high enough for dependable frame-by-frame interpretation.",
        "tip": "Improve the recording before changing technique based on this result.",
        "drill": "Record again with the entire body and club path visible, stable landscape framing, strong front lighting, and no loose clothing occluding joints.",
    },
}


def generate_advice(component_scores: Dict[str, float], summary: Dict[str, float]) -> pd.DataFrame:
    ranked = sorted(component_scores.items(), key=lambda item: item[1])
    rows = []
    for priority, (component, score) in enumerate(ranked[:4], start=1):
        item = ADVICE_LIBRARY[component]
        rows.append({
            "Priority": priority,
            "Area": component,
            "Component score": round(score, 1),
            "Insight": item["insight"],
            "Suggestion": item["tip"],
            "Practice drill": item["drill"],
        })
    return pd.DataFrame(rows)


def build_practice_plan(advice_df: pd.DataFrame, profile: Dict[str, Any]) -> pd.DataFrame:
    focus_areas = advice_df["Area"].tolist()
    sessions_per_week = int(clamp(profile.get("gym_activity_days_per_week", 3), 2, 4))
    rows = []
    for week in range(1, 5):
        primary = focus_areas[(week - 1) % len(focus_areas)]
        secondary = focus_areas[week % len(focus_areas)]
        volume = 12 + 4 * week
        for session in range(1, sessions_per_week + 1):
            rows.append({
                "Week": week,
                "Session": session,
                "Warm-up": "5-8 minutes of comfortable mobility and easy rehearsal swings",
                "Primary focus": primary,
                "Secondary focus": secondary,
                "Main drill volume": f"{volume} controlled repetitions; rest as needed",
                "Ball-striking block": f"{10 + 5 * week} balls at 50-75% effort",
                "Feedback capture": "Record the final 3 swings from the same camera position",
                "Success check": "Balanced finish, repeatable contact, and no pain",
            })
    return pd.DataFrame(rows)


advice_df = generate_advice(component_scores, swing_summary)
practice_plan_df = build_practice_plan(advice_df, USER_PROFILE)

if str(USER_PROFILE.get("health_notes", "")).strip().lower() not in {"", "none", "none reported", "n/a"}:
    print("Health note present: keep intensity conservative and obtain appropriate clearance when needed.")

display(advice_df)
display(practice_plan_df)
advice_df.to_csv(OUTPUT_DIR / "coaching_recommendations.csv", index=False)
practice_plan_df.to_csv(OUTPUT_DIR / "four_week_practice_plan.csv", index=False)

## 12. Reinforcement learning - feedback-updated contextual bandit

A contextual bandit is a practical MVP form of reinforcement learning. The current weakness is the state, each drill is an action, and the user's 1-5 rating is converted to a reward. Repeated feedback gradually changes which drill is recommended for each weakness.

For a production system, add delayed outcomes such as improvement in contact, dispersion, and independently labeled movement quality. Do not optimize only for user satisfaction.

In [ ]:
RL_ACTIONS: Dict[str, List[str]] = {
    "Posture": ["Mirror setup holds", "Alignment-stick setup checks", "Slow-motion address resets"],
    "Balance": ["Feet-together half swings", "Hold-the-finish drill", "Step-through transition drill"],
    "Rotation": ["Club-across-chest turns", "Split-stance rotation drill", "Half-swing sequencing drill"],
    "Tempo": ["Three-to-one count drill", "Metronome rehearsal", "Pause-at-the-top drill"],
    "Arm extension": ["Towel-target extension drill", "Lead-arm half swings", "Trail-hand-only soft swings"],
    "Consistency": ["Three-speed ladder", "Nine identical rehearsals", "Start-line gate drill"],
    "Pose confidence": ["Tripod recording reset", "Full-body framing check", "Lighting and clothing check"],
}
RL_POLICY_PATH = MODEL_DIR / "rl_contextual_bandit_policy.json"


class ContextualBanditCoach:
    def __init__(self, actions: Dict[str, List[str]], policy_path: Path):
        self.actions = actions
        self.policy_path = policy_path
        self.q_values = {state: {action: 0.0 for action in action_list} for state, action_list in actions.items()}
        self.counts = {state: {action: 0 for action in action_list} for state, action_list in actions.items()}
        if policy_path.exists():
            payload = json.loads(policy_path.read_text())
            for state in self.q_values:
                self.q_values[state].update(payload.get("q_values", {}).get(state, {}))
                self.counts[state].update(payload.get("counts", {}).get(state, {}))

    def recommend(self, state: str, epsilon: float = 0.10) -> str:
        available = self.actions[state]
        if random.random() < epsilon:
            return random.choice(available)
        return max(available, key=lambda action: self.q_values[state].get(action, 0.0))

    def update(self, state: str, action: str, rating_1_to_5: int) -> float:
        rating = int(clamp(rating_1_to_5, 1, 5))
        reward = (rating - 3) / 2.0  # -1, -0.5, 0, 0.5, 1
        self.counts[state][action] += 1
        count = self.counts[state][action]
        old_q = self.q_values[state][action]
        self.q_values[state][action] = old_q + (reward - old_q) / count
        self.save()
        return reward

    def save(self) -> None:
        payload = {"q_values": self.q_values, "counts": self.counts}
        self.policy_path.write_text(json.dumps(payload, indent=2))


bandit = ContextualBanditCoach(RL_ACTIONS, RL_POLICY_PATH)
RL_STATE = str(advice_df.iloc[0]["Area"])
RL_RECOMMENDATION = bandit.recommend(RL_STATE, epsilon=0.10)
print("Current state/weakness:", RL_STATE)
print("RL-selected drill:", RL_RECOMMENDATION)
print("After trying the drill, use the next cell to record feedback.")

In [ ]:
# ------------------------ OPTIONAL USER FEEDBACK ------------------------
# Set SUBMIT_FEEDBACK = True only after the user has tried the selected drill.
SUBMIT_FEEDBACK = False
FEEDBACK_RATING_1_TO_5 = 4
FEEDBACK_COMMENT = ""  # Optional local note

if SUBMIT_FEEDBACK:
    reward = bandit.update(RL_STATE, RL_RECOMMENDATION, FEEDBACK_RATING_1_TO_5)
    feedback_record = {
        "state": RL_STATE,
        "action": RL_RECOMMENDATION,
        "rating": int(FEEDBACK_RATING_1_TO_5),
        "reward": reward,
        "comment": FEEDBACK_COMMENT,
        "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    }
    feedback_path = OUTPUT_DIR / "user_feedback_history.jsonl"
    with feedback_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(feedback_record) + "\n")
    print("Feedback saved and policy updated:", feedback_record)
else:
    print("No feedback submitted. Set SUBMIT_FEEDBACK=True after the drill is tested.")

## 13. Final report, data exports, and downloadable package

The report excludes health notes by default. The complete output package contains the annotated video, frame features, summary, coaching plan, figures, models, RL policy, and an HTML report.

In [ ]:
def to_python_scalar(value: Any) -> Any:
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if pd.isna(value) if not isinstance(value, (dict, list, tuple)) else False:
        return None
    return value


def image_to_data_uri(path: Path) -> str:
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/png;base64,{encoded}"


def dataframe_html(df: pd.DataFrame) -> str:
    return df.to_html(index=False, border=0, classes="dataframe", escape=True)


profile_report = {
    key: value for key, value in USER_PROFILE.items()
    if key != "consent_to_process_video" and (INCLUDE_HEALTH_NOTES_IN_REPORT or key != "health_notes")
}
summary_serializable = {key: to_python_scalar(value) for key, value in swing_summary.items()}
components_serializable = {key: to_python_scalar(value) for key, value in component_scores.items()}
metadata_serializable = {key: to_python_scalar(value) for key, value in video_metadata.items()}

report_payload = {
    "project": "AI-Based Golf Coach",
    "student": "Isaiah Goh",
    "prototype_version": "1.0",
    "profile": profile_report,
    "video_metadata": metadata_serializable,
    "swing_summary": summary_serializable,
    "component_scores": components_serializable,
    "top_rl_state": RL_STATE,
    "rl_recommendation": RL_RECOMMENDATION,
    "limitations": [
        "Two-dimensional pose estimates depend strongly on camera position and occlusion.",
        "Synthetic data are used for ML and DL demonstration unless a real labeled CSV is supplied.",
        "The score is not a certified coaching, medical, or injury-risk assessment.",
        "Clubhead and ball-flight tracking are outside this prototype's scope.",
    ],
}

SUMMARY_JSON_PATH = OUTPUT_DIR / "golf_coach_summary.json"
SUMMARY_JSON_PATH.write_text(json.dumps(report_payload, indent=2), encoding="utf-8")

summary_table = pd.DataFrame(
    [(key.replace("_", " ").title(), round(value, 3) if isinstance(value, (int, float)) else value)
     for key, value in summary_serializable.items()],
    columns=["Metric", "Value"]
)

embedded_images = "".join(
    f'<figure><img src="{image_to_data_uri(path)}" alt="{html.escape(name)}"><figcaption>{html.escape(name.replace("_", " ").title())}</figcaption></figure>'
    for name, path in PLOT_PATHS.items() if path.exists()
)

REPORT_HTML_PATH = OUTPUT_DIR / "golf_coach_report.html"
REPORT_HTML_PATH.write_text(f"""
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Isaiah Goh - AI Golf Coach Report</title>
<style>
body {{ font-family: Arial, sans-serif; max-width: 1100px; margin: 32px auto; padding: 0 20px; line-height: 1.5; color: #1f2937; }}
h1, h2 {{ color: #12355b; }}
.notice {{ background: #fff7d6; border-left: 5px solid #c99b00; padding: 12px 16px; }}
.score {{ font-size: 34px; font-weight: bold; }}
table {{ border-collapse: collapse; width: 100%; margin: 12px 0 24px; }}
th, td {{ border-bottom: 1px solid #d1d5db; text-align: left; padding: 8px; vertical-align: top; }}
th {{ background: #eef3f8; }}
figure {{ margin: 28px 0; }}
img {{ max-width: 100%; height: auto; border: 1px solid #d1d5db; }}
.small {{ color: #4b5563; font-size: 0.92rem; }}
</style>
</head>
<body>
<h1>AI-Based Golf Coach Report</h1>
<p><strong>Student:</strong> Isaiah Goh</p>
<p class="score">{summary_serializable['heuristic_overall_score']:.1f}/100</p>
<p class="notice"><strong>Educational prototype:</strong> This score is camera-dependent and is not a certified coaching or medical assessment.</p>
<h2>Golfer Profile</h2>
{dataframe_html(pd.DataFrame(profile_report.items(), columns=['Field', 'Value']))}
<h2>Swing Summary</h2>
{dataframe_html(summary_table)}
<h2>Component Scores</h2>
{dataframe_html(pd.DataFrame(component_scores.items(), columns=['Component', 'Score']))}
<h2>Priority Coaching Suggestions</h2>
{dataframe_html(advice_df)}
<h2>Four-Week Practice Plan</h2>
{dataframe_html(practice_plan_df)}
<h2>Reinforcement Learning Recommendation</h2>
<p><strong>State:</strong> {html.escape(RL_STATE)}<br><strong>Selected action:</strong> {html.escape(RL_RECOMMENDATION)}</p>
<h2>Figures</h2>
{embedded_images}
<h2>Limitations</h2>
<ul>{''.join(f'<li>{html.escape(item)}</li>' for item in report_payload['limitations'])}</ul>
<p class="small">Generated locally by the Isaiah Goh AI Golf Coach Colab prototype.</p>
</body>
</html>
""", encoding="utf-8")

print("HTML report:", REPORT_HTML_PATH)
display(HTML(REPORT_HTML_PATH.read_text(encoding="utf-8")))

In [ ]:
# Save a compact model card and README for responsible reuse.
MODEL_CARD_PATH = OUTPUT_DIR / "MODEL_CARD.md"
MODEL_CARD_PATH.write_text("""# Isaiah Goh AI Golf Coach - Model Card

## Intended use
Educational demonstration of pose estimation, feature extraction, supervised learning,
sequence deep learning, feedback-based recommendation, and report generation.

## Not intended for
Medical diagnosis, injury-risk prediction, certified coaching decisions, talent selection,
or autonomous high-stakes recommendations.

## Training data
The default Random Forest and GRU use synthetic data generated inside the notebook.
They must be retrained and validated with de-identified, consented, coach-labeled data.

## Main limitations
- 2D camera perspective and occlusion can distort angles.
- Clothing, lighting, frame rate, and camera motion affect landmark quality.
- The club and ball are not tracked.
- Demographic and ability-group fairness have not been evaluated.
- User feedback can encode preference bias and should not be the only RL objective.
""", encoding="utf-8")

README_PATH = OUTPUT_DIR / "README_OUTPUTS.txt"
README_PATH.write_text("""Isaiah Goh AI Golf Coach output package

Key files:
- isaiah_goh_golf_swing_annotated.mp4: annotated input video
- golf_coach_report.html: self-contained report
- swing_frame_features.csv: frame-level measurements
- coaching_recommendations.csv: prioritized feedback
- four_week_practice_plan.csv: personalized plan
- golf_coach_summary.json: machine-readable summary
- MODEL_CARD.md: intended use and limitations
- models/: Random Forest, GRU, and contextual-bandit policy files

Synthetic-data warning:
The default ML and DL models demonstrate architecture only. Do not report their
metrics as real-world golf coaching accuracy.
""", encoding="utf-8")

FINAL_ZIP_PATH = BASE_DIR / "Isaiah_Goh_AI_Golf_Coach_Outputs.zip"
with zipfile.ZipFile(FINAL_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in OUTPUT_DIR.rglob("*"):
        if file_path.is_file():
            archive.write(file_path, arcname=f"outputs/{file_path.relative_to(OUTPUT_DIR)}")
    for file_path in MODEL_DIR.rglob("*"):
        if file_path.is_file():
            archive.write(file_path, arcname=f"models/{file_path.relative_to(MODEL_DIR)}")

print("Final output package:", FINAL_ZIP_PATH)
print("Package size (MB):", round(FINAL_ZIP_PATH.stat().st_size / (1024 ** 2), 2))

if IN_COLAB and AUTO_DOWNLOAD_FINAL_ZIP:
    files.download(str(FINAL_ZIP_PATH))
else:
    print("To download in Colab, run: files.download(str(FINAL_ZIP_PATH))")

## 14. Production roadmap

A stronger version should add:

1. **Data governance:** explicit consent, age-appropriate terms, data minimization, encryption, retention controls, deletion, and de-identification.
2. **Labeled data:** multiple qualified coaches, repeated labels, diverse body types, ability levels, camera views, clubs, and shot types.
3. **Evaluation:** subject-level train/test separation, cross-camera validation, calibration, uncertainty, subgroup analysis, and inter-rater reliability.
4. **Computer vision:** club and ball detection, 3D pose, multi-view calibration, temporal smoothing, and automatic swing-event detection.
5. **Personalization:** distinguish technical goals, physical limitations, practice frequency, and skill level without making medical diagnoses.
6. **Reinforcement learning safeguards:** optimize objective improvement plus user satisfaction, constrain unsafe recommendations, and allow coach override.
7. **Application layer:** Streamlit/FastAPI front end, authenticated profiles, object storage, asynchronous video jobs, database audit trails, and model monitoring.
8. **Human-in-the-loop review:** let a professional coach inspect the video, measurements, confidence, and generated practice plan.

### Suggested real-data table

One row per swing should include anonymized user ID, session ID, camera view, handedness, club, shot type, 33-landmark sequence or derived features, coach phase labels, component scores, overall score, feedback, and follow-up outcome.

## References used for the implementation approach

- Google Colab FAQ: hosted Jupyter notebooks with Python and optional accelerators.
- Google AI Edge MediaPipe Pose Landmarker Python API: pose landmarks and video-mode inference.
- scikit-learn `RandomForestRegressor`: ensemble regression for supervised scoring.
- PyTorch `torch.nn.GRU`: gated recurrent units for ordered swing sequences.

The notebook intentionally uses official APIs and transparent local logic; it does not require a paid API key.